# Student Performance Prediction & Data Drift Detection
### End-to-End MLOps Pipeline with MLflow and Evidently

**Author:** Senior MLOps Engineer & Project Mentor  
**Dataset:** UCI Student Performance (Mathematics)  
**Objective:** Predict final mathematics grades (`G3`) using student background attributes, track experiments with MLflow, and monitor data drift with Evidently.

## 1. Problem Statement & Architecture

Educational institutions seek early indicators of student performance to provide timely academic support. This project develops a supervised machine learning regression model to predict a student's final mathematics grade (`G3`) based purely on demographic, family background, and study habits.

> **Important Design Choice:** Earlier exam grades (`G1` and `G2`) are deliberately **excluded** as model input features to evaluate whether background and habit indicators alone can predict final achievement.

Furthermore, machine learning models in production face **data drift** when the incoming student population's distributions shift over time. We simulate controlled data drift and detect it using **Evidently**.

## 2. Environment Setup & Library Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path to allow modular imports from src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_DATA_PATH,
    TARGET_COLUMN,
    EXCLUDE_COLUMNS,
    MONITORED_FEATURES,
    BASELINE_MODEL_PARAMS,
    RUN2_MODEL_PARAMS
)
from src.data_loader import load_and_validate_data
from src.train import run_training_pipeline
from src.prepare_data import prepare_and_save_drift_datasets
from src.detect_drift import run_drift_pipeline

print("✓ Imports and configurations loaded successfully!")

## 3. Dataset Loading, Inspection & Quality Checks

We load `student-mat.csv` using the semicolon delimiter (`;`), validate the schema, check for missing values, and inspect data types.

In [ ]:
df, quality_summary = load_and_validate_data()
print(f"Dataset Dimensions: {df.shape[0]} records, {df.shape[1]} columns")
df.head()

In [ ]:
# Statistical summary of the target variable G3
print("Target G3 Distribution Summary:")
print(df[TARGET_COLUMN].describe())

## 4. Exploratory Data Analysis (EDA)

Let us explore how key features like `studytime`, `failures`, and `absences` correlate with the final mathematics grade `G3`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# G3 vs Study Time
sns.boxplot(data=df, x="studytime", y="G3", ax=axes[0], palette="Blues")
axes[0].set_title("Final Grade (G3) vs. Weekly Study Time")

# G3 vs Past Failures
sns.boxplot(data=df, x="failures", y="G3", ax=axes[1], palette="Reds")
axes[1].set_title("Final Grade (G3) vs. Past Failures")

# G3 vs Absences
sns.scatterplot(data=df, x="absences", y="G3", ax=axes[2], color="#2b5c8f", alpha=0.6)
axes[2].set_title("Final Grade (G3) vs. School Absences")

plt.tight_layout()
plt.show()

## 5. Feature & Target Selection and Pipeline Preprocessing

- **Target:** `G3` (final exam grade, range 0–20)
- **Excluded:** `G1` and `G2` (prior period grades) and `G3`
- **Numerical pipeline:** `SimpleImputer(strategy='median')`
- **Categorical pipeline:** `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')`

## 6. Model Training & MLflow Experiment Tracking

We train two reproducible configurations of `RandomForestRegressor`:
1. **Run 1 (Baseline):** `n_estimators=100`, `max_depth=8`
2. **Run 2 (Comparative):** `n_estimators=200`, `max_depth=12`

Both runs are recorded into the local MLflow experiment `Student_Performance_Prediction`.

In [ ]:
best_pipeline, training_summary = run_training_pipeline()
print("Selected Model:", training_summary["selected_model"])
print("Evaluation Metrics:", training_summary["metrics"])

## 7. Reference & Current Dataset Preparation (Simulated Drift)

To test our monitoring system, we create:
- **Reference Dataset (`data/reference_data.csv`):** 150 sampled records from baseline data.
- **Current Dataset (`data/current_data.csv`):** A copy of the reference data with 3 controlled, deterministic transformations:
  1. `studytime`: Shifted categories 1 & 2 up to 4.
  2. `absences`: Uniformly increased by +5.
  3. `failures`: Incremented by +1 (capped at 4).

In [ ]:
ref_df, curr_df, drift_summary = prepare_and_save_drift_datasets()
print(f"Reference Dataset Shape: {ref_df.shape}")
print(f"Current Dataset Shape:   {curr_df.shape}")

## 8. Evidently Data Drift Detection

We use **Evidently** to evaluate statistical drift across 6 features (`studytime`, `absences`, `failures`, `age`, `Medu`, `Fedu`).
Reports are generated in both **HTML** and **JSON** formats.

In [ ]:
drift_results, pred_results = run_drift_pipeline()
print(f"Dataset-Level Drift Detected: {drift_results['dataset_drift']}")
print(f"Drifted Features: {drift_results['drifted_features_count']} / {drift_results['total_monitored_features']}")

## 9. Prediction Drift Analysis

We also monitor how our trained model responds to the shifted data distribution. Even though the model itself hasn't changed, its predicted grade distribution shifts in response to increased absences and failures.

In [ ]:
if pred_results:
    print("Reference Predicted G3 Mean:", pred_results['reference_predictions']['mean'])
    print("Current Predicted G3 Mean:  ", pred_results['current_predictions']['mean'])
    print("Net Prediction Shift:       ", pred_results['prediction_mean_shift'])

## 10. Conclusion & Recommendations

1. **Predictive Capability:** Demographic and study habit variables provide meaningful baseline signals for mathematics achievement, though predicting human performance purely from background exhibits expected residual variance.
2. **Experiment Tracking:** MLflow provides transparency and auditability, recording exact hyperparameters and metric artifacts.
3. **Drift Monitoring:** Evidently successfully identified statistically significant distribution shifts in the modified features without false positives on unchanged demographic baseline variables.
4. **Operational Action:** When dataset-level drift exceeds thresholds in production, data teams must investigate source demographic shifts, trigger automated retraining, and notify academic counselors.